In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [14]:
# read dataset
df = pd.read_csv('../data/SGJobData.csv.xz')
df.head()

,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary
0,"[{""id"":13,""category"":""Environment / Health""},{...",Permanent,2023-05-08,False,MCF-2023-0252866,2023-04-08,2023-03-30,2,5,151,...,NaN,Executive,WORKSTONE PTE. LTD.,2800,2000,Monthly,0,Closed,Food Technologist - Clementi | Entry Level | U...,2400.0
1,"[{""id"":21,""category"":""Information Technology""}]",Permanent,2023-05-08,False,MCF-2023-0273977,2023-04-08,2023-04-08,0,0,55,...,NaN,Executive,TRUST RECRUIT PTE. LTD.,5500,4000,Monthly,0,Closed,"Software Engineer (Fab Support) (Java, CIM, Up...",4750.0
2,"[{""id"":33,""category"":""Repair and Maintenance""}]",Full Time,2023-04-22,False,MCF-2023-0273994,2023-04-08,2023-04-08,0,7,99,...,NaN,Senior Executive,PU TIEN SERVICES PTE. LTD.,4600,3800,Monthly,0,Closed,Senior Technician,4200.0
3,"[{""id"":21,""category"":""Information Technology""}]",Permanent,2023-05-08,False,MCF-2023-0273991,2023-04-08,2023-04-08,0,6,113,...,NaN,Senior Executive,TRUST RECRUIT PTE. LTD.,10000,5000,Monthly,0,Closed,"Senior .NET Developer (.NET Core, MVC, MVVC, S...",7500.0
4,"[{""id"":2,""category"":""Admin / Secretarial""}]",Full Time,2023-05-08,False,MCF-2023-0273976,2023-04-08,2023-04-08,0,3,99,...,NaN,Non-executive,EATZ CATERING SERVICES PTE. LTD.,3400,2400,Monthly,0,Closed,Sales / Admin Cordinator,2900.0


In [15]:
#remove columns "employment_type", "metadata_isPostedOnBehalf", "metadata_jobPostId", "salary_maximum", "Salary_minimum" 
df = df.drop(columns=["employmentTypes","metadata_expiryDate", "metadata_isPostedOnBehalf", "metadata_jobPostId", "metadata_newPostingDate", "salary_maximum", "salary_minimum", "occupationId", "status_id","average_salary"])
df.head()

,categories,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,minimumYearsExperience,numberOfVacancies,positionLevels,postedCompany_name,salary_type,status_jobStatus,title
0,"[{""id"":13,""category"":""Environment / Health""},{...",2023-03-30,2,5,151,0,1,Executive,WORKSTONE PTE. LTD.,Monthly,Closed,Food Technologist - Clementi | Entry Level | U...
1,"[{""id"":21,""category"":""Information Technology""}]",2023-04-08,0,0,55,2,2,Executive,TRUST RECRUIT PTE. LTD.,Monthly,Closed,"Software Engineer (Fab Support) (Java, CIM, Up..."
2,"[{""id"":33,""category"":""Repair and Maintenance""}]",2023-04-08,0,7,99,3,1,Senior Executive,PU TIEN SERVICES PTE. LTD.,Monthly,Closed,Senior Technician
3,"[{""id"":21,""category"":""Information Technology""}]",2023-04-08,0,6,113,8,1,Senior Executive,TRUST RECRUIT PTE. LTD.,Monthly,Closed,"Senior .NET Developer (.NET Core, MVC, MVVC, S..."
4,"[{""id"":2,""category"":""Admin / Secretarial""}]",2023-04-08,0,3,99,2,3,Non-executive,EATZ CATERING SERVICES PTE. LTD.,Monthly,Closed,Sales / Admin Cordinator


In [ ]:
df.info()

In [ ]:
def explode_categories(df, col="categories"):
    out = df.copy()

    out[col] = (
        out[col]
        .dropna()
        .map(lambda x: json.loads(x) if isinstance(x, str) else x)
    )

    out = out.explode(col, ignore_index=True)

    out["category_id"] = out[col].map(lambda d: d.get("id") if isinstance(d, dict) else None)
    out["category"]    = out[col].map(lambda d: d.get("category") if isinstance(d, dict) else None)

    return out.drop(columns=[col])

df_exploded = explode_categories(df)
df_exploded.head()

In [ ]:
# if company name contains 'kpmg', show list of such rows
df_exploded[df_exploded['postedCompany_name'].str.contains('kpmg', case=False, na=False)].sort_values('metadata_originalPostingDate', ascending=False)


In [ ]:
df_exploded.category.value_counts()

In [ ]:
df_exploded[df_exploded['category'] == 'Accounting / Auditing / Taxation'][['category','postedCompany_name','metadata_jobPostId','numberOfVacancies']] \
    .groupby('postedCompany_name').numberOfVacancies.sum().sort_values(ascending=False)